# Hinglish, second pass: corrected Qwen format, stronger Roman-script instruction

The baseline (`compare-hinglish-2models.ipynb`, same branch) found the pipeline correctly configured, but neither model usable: Qwen + adapter wrote Roman Hinglish that never stopped, drifted and looped; Gemma 3 stopped cleanly and read well but answered three of five prompts in Devanagari and one in plain English. This pass changes only the prompt format, as the next-experiment note asks:

| | Baseline | This pass |
|---|---|---|
| **Qwen2.5-1.5B + Hinglish adapter** | instruction as a **system** turn | instruction at the start of the **user** turn: the adapter's documented format is plain ChatML **without** a system prompt, which is how it was trained. Same stop tokens (`<|im_end|>`, `<|endoftext|>`) |
| **Gemma 3 1B IT** | its official template, the document's instruction | its official template (unchanged), with a **stronger Roman-script instruction**: no Devanagari at all, not pure English or pure Hindi, and one short example sentence in Roman Hinglish (unrelated to the five prompts) |
| Both | | the same strengthened instruction text; every output records **how it ended**: on EOS (which token), at the 80-token limit, or otherwise, and whether the text ends at a sentence boundary |

Unchanged from the baseline: the five prompts, greedy decoding, at most 80 new tokens, no repetition penalty, fp32 on a T4, seed 42, pinned revisions, and the printed prompts, raw decodes and token IDs. The flags are diagnostics, not quality scores; the rubric is a human review.

**Runs on:** Kaggle, GPU T4, Internet on. Paste a Hugging Face token with access to `google/gemma-3-1b-it` into Cell 0, and clear it again before saving.

In [ ]:
# Cell 0 — Hugging Face token: paste it here before running (needed for the gated Gemma 3)
# Clear it again before saving or sharing the notebook, so the token never ends up in a file.

HF_TOKEN = ""

In [2]:
# Cell 1 — Setup: packages, settings, versions

!pip uninstall -y -q torchao                 # Kaggle ships torchao 0.10; peft refuses versions below 0.16
!pip install -q "transformers==5.17.0" "peft==0.21.2"

import json, re, time

import peft
import torch
import transformers
from huggingface_hub import HfApi, login

MAX_NEW_TOKENS = 80
SEED = 42
DTYPE = torch.float32                        # a T4 has no fast bfloat16, and Gemma 3 is unstable in fp16
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
HF_TOKEN = HF_TOKEN.strip() or None
if HF_TOKEN:
    login(HF_TOKEN)
VERSIONS = {"torch": torch.__version__, "transformers": transformers.__version__, "peft": peft.__version__,
            "device": torch.cuda.get_device_name(DEVICE) if DEVICE.type == "cuda" else "cpu", "dtype": str(DTYPE)}
print(VERSIONS, "| HF token:", "yes" if HF_TOKEN else "no (Gemma 3 is gated and will fail)")

{'torch': '2.11.0+cu128', 'transformers': '5.17.0', 'peft': '0.21.2', 'device': 'Tesla T4', 'dtype': 'torch.float32'} | HF token: yes


## Cell 2 — The instruction and the five prompts

The comparison document's instruction, word for word, plus three stricter script rules with one example sentence (this pass). The five prompts are unchanged. Both models get the same instruction text; only where it goes differs (Cell 4).

In [3]:
# Cell 2 — The instruction (the document's + stricter script rules) and the five prompts

SYSTEM = """You write natural, conversational Hinglish using the Roman/Latin alphabet.

Follow these rules:
- Write Hindi words in Roman script, mixed naturally with English where appropriate.
- Do not use Devanagari.
- Do not explain your process or repeat the instructions.
- Respond directly to the scenario in 1–3 short sentences.
- Keep the wording natural and readable, like something a person would actually say.
- Do not repeat phrases or sentences.
- Do not add labels, headings, quotation marks, or a character name unless the scenario asks for them.
- Do not copy dialogue from films or other copyrighted sources.

Script rules, strictly:
- Use only the Roman/Latin alphabet. Never use Devanagari characters, not even for a single Hindi word.
- Do not answer in pure English or in pure Hindi: mix Hindi and English the way people text each other in India.
- Example of the style (not an answer to use): Arre yaar, traffic itna tha ki meeting ke liye late ho gaya."""

PROMPTS = {
    "P1 casual": "Scenario: Abhee chai ka mood hai, but kaam bhi pending hai.\n"
                 "Write a short, natural response that a person might say in this situation.",
    "P2 celebration": "Scenario: Aaj Lumiq company ke 13th anniversary ki party hai, and the whole team is excited.\n"
                      "Write a short, natural Hinglish response that fits this moment.",
    "P3 boring day": "Scenario: Today was so boring that I counted the ceiling fan's rotations.\n"
                     "Write a witty but natural Hinglish response. Do not translate the scenario literally.",
    "P4 credit score": "Scenario: My friend is worried because their credit score dropped.\n"
                       "Write a short, empathetic Hinglish response that sounds conversational, not like a formal article.",
    "P5 fed up": "Scenario: It happened again, and this time I am really fed up.\n"
                 "Write a short, natural Hinglish response that conveys frustration without over-explaining.",
}
print(f"system instruction: {len(SYSTEM)} characters | prompts: {list(PROMPTS)}")

system instruction: 962 characters | prompts: ['P1 casual', 'P2 celebration', 'P3 boring day', 'P4 credit score', 'P5 fed up']


## Cell 3 — Verify the repositories and pin them

Each repository is pinned to the revision found now. For the adapter: its configured base model must be exactly `Qwen/Qwen2.5-1.5B`, and the base's latest revision must predate the adapter's first commit (so the adapter was trained on the revision we load).

In [4]:
# Cell 3 — Verify and pin: base revision, adapter base model and timing, Gemma access

api = HfApi()
BASE, ADAPTER, GEMMA = "Qwen/Qwen2.5-1.5B", "sarthaksenapati/qwen1.5b-hinglish-sft-v2", "google/gemma-3-1b-it"
REV = {r: api.model_info(r, token=HF_TOKEN).sha for r in (BASE, ADAPTER, GEMMA)}

adapter_cfg = json.load(open(api.hf_hub_download(ADAPTER, "adapter_config.json", revision=REV[ADAPTER])))
base_commits = api.list_repo_commits(BASE, token=HF_TOKEN)
adapter_commits = api.list_repo_commits(ADAPTER, token=HF_TOKEN)
base_latest, adapter_first = base_commits[0].created_at, adapter_commits[-1].created_at
CHECKS = {
    "adapter base model": adapter_cfg["base_model_name_or_path"],
    "adapter targets this base": adapter_cfg["base_model_name_or_path"] == BASE,
    "base latest revision": f"{REV[BASE][:12]} ({base_latest:%Y-%m-%d})",
    "adapter first commit": f"{adapter_first:%Y-%m-%d}",
    "base unchanged since before the adapter": base_latest < adapter_first,
    "adapter": {k: adapter_cfg.get(k) for k in ("peft_type", "r", "lora_alpha", "lora_dropout", "target_modules")},
}
for k, v in CHECKS.items():
    print(f"{k:42s} {v}")
for r, s in REV.items():
    print(f"pinned {r:44s} {s}")
assert CHECKS["adapter targets this base"] and CHECKS["base unchanged since before the adapter"], "adapter / base mismatch"

adapter base model                         Qwen/Qwen2.5-1.5B
adapter targets this base                  True
base latest revision                       8faed761d45a (2024-10-08)
adapter first commit                       2026-06-21
base unchanged since before the adapter    True
adapter                                    {'peft_type': 'LORA', 'r': 32, 'lora_alpha': 64, 'lora_dropout': 0.05, 'target_modules': ['gate_proj', 'v_proj', 'k_proj', 'o_proj', 'q_proj', 'down_proj', 'up_proj']}
pinned Qwen/Qwen2.5-1.5B                            8faed761d45a263340a0528343f099c05c9a4323
pinned sarthaksenapati/qwen1.5b-hinglish-sft-v2     942fd0b2ebd2119d772f18952de6f8d372a6448b
pinned google/gemma-3-1b-it                         dcc83ea841ab6100d6b47a070329e1ba4cf78752


## Cell 4 — Load, build the prompt, generate

- **Qwen + adapter:** the base model, then `PeftModel.from_pretrained` with the adapter (its documented method); the tokenizer comes from the adapter repository, with its chat template. Stop tokens: `<|im_end|>` and `<|endoftext|>`.
- **Gemma 3:** its model and tokenizer; stop tokens from its generation config (`<eos>`, `<end_of_turn>`).
- The prompt is `[system, user]` through each model's template with `add_generation_prompt=True`, tokenized **without** adding extra special tokens (the template already holds any start token). Only the generated part is decoded: once cleaned (`skip_special_tokens=True`) and once raw, for token inspection.

In [5]:
# Cell 4 — Load each model with its own template and stop tokens; generate greedily

from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer


def load(name):
    if name == "qwen2.5-1.5b + hinglish adapter":
        tok = AutoTokenizer.from_pretrained(ADAPTER, revision=REV[ADAPTER])
        base = AutoModelForCausalLM.from_pretrained(BASE, revision=REV[BASE], dtype=DTYPE).to(DEVICE)
        model = PeftModel.from_pretrained(base, ADAPTER, revision=REV[ADAPTER])
        stops = [tok.convert_tokens_to_ids("<|im_end|>"), tok.convert_tokens_to_ids("<|endoftext|>")]
    else:
        tok = AutoTokenizer.from_pretrained(GEMMA, revision=REV[GEMMA], token=HF_TOKEN)
        model = AutoModelForCausalLM.from_pretrained(GEMMA, revision=REV[GEMMA], dtype=DTYPE, token=HF_TOKEN).to(DEVICE)
        eos = model.generation_config.eos_token_id
        stops = list(eos) if isinstance(eos, list) else [eos]
    model.eval()
    return tok, model, sorted(set(stops))


def messages(name, user):
    """Qwen + adapter: no system turn (its documented, trained format), the instruction opens the user turn.
    Gemma: system + user through its official template (which folds the system text into the user turn)."""
    if name.startswith("qwen"):
        return [{"role": "user", "content": f"{SYSTEM}\n\n{user}"}]
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]


def generate(tok, model, stops, name, user):
    text = tok.apply_chat_template(messages(name, user), add_generation_prompt=True, tokenize=False)
    ids = tok(text, return_tensors="pt", add_special_tokens=False).to(DEVICE)
    torch.manual_seed(SEED)
    t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**ids, do_sample=False, num_beams=1, max_new_tokens=MAX_NEW_TOKENS, eos_token_id=stops,
                             pad_token_id=tok.pad_token_id if tok.pad_token_id is not None else stops[0],
                             temperature=None, top_p=None, top_k=None)
    new = out[0, ids["input_ids"].shape[1]:].tolist()
    output = tok.decode(new, skip_special_tokens=True).strip()
    if new and new[-1] in stops:
        ended = f"EOS ({tok.convert_ids_to_tokens(new[-1])})"
    elif len(new) >= MAX_NEW_TOKENS:
        ended = "token limit"
    else:
        ended = "other"
    return {"prompt": text, "output": output, "raw": tok.decode(new), "token_ids": new, "new_tokens": len(new),
            "stopped_cleanly": bool(new) and new[-1] in stops, "ended": ended,
            "ends_at_sentence": output.rstrip("”\"' ").endswith((".", "!", "?", "।")),
            "ms": round((time.perf_counter() - t0) * 1000)}

---
## Cell 5 — Run both models on the five prompts

The full prompt each model received is printed once (from the first prompt), then every output. Diagnostic flags per output: Devanagari share (should be 0), clean stop (ended on a stop token before 80 tokens), the most repeated word 3-gram (> 2 suggests a loop), and leakage (the output repeats a line of the instruction or the scenario).

In [6]:
# Cell 5 — Generate, then print prompts, outputs, raw decodes, token IDs and diagnostic flags

DEVA = re.compile(r"[ऀ-ॿ]")
LATN = re.compile(r"[A-Za-z]")
LEAK_LINES = [l.strip("- ").lower() for l in SYSTEM.splitlines() if len(l) > 25] + ["scenario:", "write a short", "traffic itna tha"]


def flags(r, user):
    t = r["output"]
    d, l = len(DEVA.findall(t)), len(LATN.findall(t))
    words = t.split()
    grams = [" ".join(words[i:i + 3]) for i in range(len(words) - 2)]
    low = t.lower()
    return {"devanagari_share": round(d / (d + l), 3) if d + l else None, "stopped_cleanly": r["stopped_cleanly"],
            "top_trigram_repeats": max((grams.count(g) for g in set(grams)), default=0),
            "leaks_instruction_or_scenario": any(x[:40] in low for x in LEAK_LINES) or user.split("\n")[0][10:60].lower() in low}


RESULTS = []
for name in ["qwen2.5-1.5b + hinglish adapter", "gemma-3-1b-it"]:
    t0 = time.perf_counter()
    tok, model, stops = load(name)
    print(f"\n===== {name} | loaded in {time.perf_counter() - t0:.0f}s | stop token IDs {stops}")
    for pid, user in PROMPTS.items():
        r = generate(tok, model, stops, name, user)
        r.update({"model": name, "prompt_id": pid, "user": user, "flags": flags(r, user),
                  "revisions": {k: REV[k] for k in ([BASE, ADAPTER] if "qwen" in name else [GEMMA])}})
        RESULTS.append(r)
        if pid == "P1 casual":
            print(f"--- full prompt as the model sees it:\n{r['prompt']}\n--- end of prompt")
        print(f"\n[{pid}] {r['new_tokens']} tokens, {r['ms']} ms | ended: {r['ended']} | ends at a sentence: {r['ends_at_sentence']} | flags {r['flags']}")
        print(f"  output: {r['output']}")
        print(f"  raw   : {r['raw']!r}")
        print(f"  ids   : {r['token_ids']}")
    del model
    torch.cuda.empty_cache() if DEVICE.type == "cuda" else None

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


===== qwen2.5-1.5b + hinglish adapter | loaded in 6s | stop token IDs [151643, 151645]
--- full prompt as the model sees it:
<|im_start|>user
You write natural, conversational Hinglish using the Roman/Latin alphabet.

Follow these rules:
- Write Hindi words in Roman script, mixed naturally with English where appropriate.
- Do not use Devanagari.
- Do not explain your process or repeat the instructions.
- Respond directly to the scenario in 1–3 short sentences.
- Keep the wording natural and readable, like something a person would actually say.
- Do not repeat phrases or sentences.
- Do not add labels, headings, quotation marks, or a character name unless the scenario asks for them.
- Do not copy dialogue from films or other copyrighted sources.

Script rules, strictly:
- Use only the Roman/Latin alphabet. Never use Devanagari characters, not even for a single Hindi word.
- Do not answer in pure English or in pure Hindi: mix Hindi and English the way people text each other in India.
- 

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]


===== gemma-3-1b-it | loaded in 5s | stop token IDs [1, 106]
--- full prompt as the model sees it:
<bos><start_of_turn>user
You write natural, conversational Hinglish using the Roman/Latin alphabet.

Follow these rules:
- Write Hindi words in Roman script, mixed naturally with English where appropriate.
- Do not use Devanagari.
- Do not explain your process or repeat the instructions.
- Respond directly to the scenario in 1–3 short sentences.
- Keep the wording natural and readable, like something a person would actually say.
- Do not repeat phrases or sentences.
- Do not add labels, headings, quotation marks, or a character name unless the scenario asks for them.
- Do not copy dialogue from films or other copyrighted sources.

Script rules, strictly:
- Use only the Roman/Latin alphabet. Never use Devanagari characters, not even for a single Hindi word.
- Do not answer in pure English or in pure Hindi: mix Hindi and English the way people text each other in India.
- Example of the sty

## Cell 6 — Side by side, and the records for the human review

The two models' outputs per prompt, a summary of the diagnostic flags (**not** quality scores), and every result as JSON lines between markers, so the result file and the scoring sheet can be rebuilt from the saved notebook. The rubric (naturalness, scenario relevance, grammar/readability, Roman-script compliance, code-switching, repetition, instruction following; 1–5 each) is filled in by a human reviewer.

In [7]:
# Cell 6 — Outputs side by side, flag summary, and the full records

for pid in PROMPTS:
    print(f"\n=== {pid}: {PROMPTS[pid].splitlines()[0]}")
    for r in [r for r in RESULTS if r["prompt_id"] == pid]:
        print(f"  {r['model']:32s} | {r['output']}")

print("\nDIAGNOSTIC FLAGS ONLY (not quality scores)")
for name in dict.fromkeys(r["model"] for r in RESULTS):
    rs = [r for r in RESULTS if r["model"] == name]
    print(f"  {name:32s} | Devanagari in {sum((r['flags']['devanagari_share'] or 0) > 0 for r in rs)} / 5 | clean stop "
          f"{sum(r['stopped_cleanly'] for r in rs)} / 5 | possible loops {sum(r['flags']['top_trigram_repeats'] > 2 for r in rs)} / 5 | "
          f"leakage {sum(r['flags']['leaks_instruction_or_scenario'] for r in rs)} / 5 | median tokens {sorted(r['new_tokens'] for r in rs)[2]}")
    print(f"  {'':32s} | ended: {dict((e, sum(r['ended'] == e for r in rs)) for e in dict.fromkeys(r['ended'] for r in rs))} | "
          f"ends at a sentence {sum(r['ends_at_sentence'] for r in rs)} / 5")

print("\n----- SETTINGS -----")
print(json.dumps({"versions": VERSIONS, "revisions": REV, "checks": {k: str(v) for k, v in CHECKS.items()},
                  "generation": {"do_sample": False, "num_beams": 1, "max_new_tokens": MAX_NEW_TOKENS, "repetition_penalty": None,
                                 "seed": SEED, "qwen_template": "adapter chat template (plain ChatML), no system turn: instruction opens the user turn",
                                 "gemma_template": "gemma-3-1b-it chat template, system + user", "instruction": "document's + stricter script rules + one example"}}, ensure_ascii=False))
print("----- RESULTS JSONL BEGIN -----")
for r in RESULTS:
    print(json.dumps({k: v for k, v in r.items() if k != "prompt"}, ensure_ascii=False))
print("----- RESULTS JSONL END -----")


=== P1 casual: Scenario: Abhee chai ka mood hai, but kaam bhi pending hai.
  qwen2.5-1.5b + hinglish adapter  | Abhee chai ka mood hai, lekin kaam bhi pending hai. Aapko ek baar chai lena padega, lekin uske baad aapko apne tasks complete karne ki koshish karna hoga. Agar aapko time nahi mil raha, toh ek baar chai lena aur phir tasks complete karne ki
  gemma-3-1b-it                    | अरे यार, चाय का मूड है, पर काम भी अभी बाकी है।

=== P2 celebration: Scenario: Aaj Lumiq company ke 13th anniversary ki party hai, and the whole team is excited.
  qwen2.5-1.5b + hinglish adapter  | Aaj Lumiq company ke 13th anniversary ki party hai, aur sabse logon ko bahut mazedaar hain. Aapke paas kuch special karna hai, ya phir kuch karte hain, ya phir kuch share karte hain. Aapka reaction natural aur natural Hinglish se likhna chahiye
  gemma-3-1b-it                    | Wow, that’s a huge milestone! So happy for Lumiq, isn’t it? Let’s celebrate!

=== P3 boring day: Scenario: Today was so boring th

---
## Next

Save the notebook (clear the token in Cell 0 first). Then the ten outputs are reviewed with the seven-criterion rubric next to the baseline's, and only then is the choice made: Qwen, Gemma, or another candidate.